# SHAP API Notebook


This notebook demonstrates how to interact with the CoinGecko API both directly and through custom Python wrapper functions. These wrapper functions are currently implemented in `src/ingestion/fetch_data.py` and `src/ingestion/dataset_builder.py`, and will later be consolidated into `SHAP_utils.py` for cleaner integration.

The purpose of this notebook is to:
- Show how raw CoinGecko endpoints can be accessed using `requests`
- Illustrate how our wrapper functions abstract and simplify the API interaction
- Validate the output format and content of the returned data

This notebook is tightly linked with `SHAP.API.md`, which documents:
- The native CoinGecko API endpoints used in this project
- The design and behavior of the software layer (wrappers) implemented on top
- Input/output structures for each function and their purpose in the broader pipeline

Unlike the full pipeline tutorial (`SHAP.example.ipynb`), this notebook is minimal and focused only on demonstrating API and wrapper functionality — not on modeling or SHAP analysis.


In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Imports and Setup

In [7]:
# Add src/ to path

import sys
sys.path.append("src")

In [8]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv


# Load API key from .env
load_dotenv()

# Custom wrapper import
from ingestion.fetch_data import fetch_market_chart_data


## 2. COINGECKO API Request (Direct requests Usage) - Market Chart (Raw)


We start by using the `/coins/{id}/market_chart` endpoint to get historical price, market cap, and volume data.

This example shows how to access the raw CoinGecko API using `requests`.


In [9]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
params = {
    "vs_currency": "usd",
    "days": 7
}
headers = {
    "accept": "application/json",
    "x-cg-demo-api-key": os.getenv("COINGECKO_API_KEY")
}

response = requests.get(url, params=params, headers=headers)
response.raise_for_status()
data = response.json()

# Show first 2 price entries
data["prices"][:2]


[[1745535705573, 93463.48432069764], [1745539602330, 93950.39664992713]]

## 3. Wrapper Call – Market Chart (Structured)

Instead of repeating raw request logic, we use a wrapper function `fetch_market_chart_data()` which:
- Handles authentication
- Formats and merges data
- Returns a clean DataFrame

In [11]:
# Mimicking config structure (without loading YAML)
config = {
    "api": {
        "base_url": url,
        "vs_currency": "usd",
        "days": 7
    }
}

df = fetch_market_chart_data(config)
df.head()

,timestamp,price,market_cap,volume
0,2025-04-24 23:01:45.573,93463.484321,1.855783e+12,2.912783e+10
1,2025-04-25 00:06:42.330,93950.396650,1.864857e+12,2.864077e+10
2,2025-04-25 01:09:27.251,93937.732911,1.865339e+12,3.054681e+10
3,2025-04-25 02:02:10.684,93724.292162,1.861160e+12,3.140209e+10
4,2025-04-25 03:07:12.949,93056.129837,1.847545e+12,3.229708e+10


## 4. Explore Other Endpoints: `/coins/list`

This endpoint returns a list of all supported coins on CoinGecko. Useful for dynamic UIs, validation, or exploratory workflows.

In [12]:
url = "https://api.coingecko.com/api/v3/coins/list"
response = requests.get(url)
coins = response.json()
pd.DataFrame(coins).head()

,id,symbol,name
0,_,gib,༼ つ ◕_◕ ༽つ
1,01coin,zoc,01coin
2,0chain,zcn,Zus
3,0dog,0dog,Bitcoin Dogs
4,0-knowledge-network,0kn,0 Knowledge Network


## 5. Endpoint: `/simple/price` – Real-Time Quotes

A lightweight endpoint to get current price of one or more assets, without historical data.

In [13]:
url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": "bitcoin,ethereum,solana", "vs_currencies": "usd"}
response = requests.get(url, params=params)
response.json()

{'bitcoin': {'usd': 96463},
 'ethereum': {'usd': 1839.77},
 'solana': {'usd': 150.29}}

## 6. Endpoint: `/coins/{id}` – Metadata and Description

This endpoint gives comprehensive details about a coin, including website, description, image, and other metadata.

In [14]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin"
response = requests.get(url)
coin_meta = response.json()

coin_meta["links"]["homepage"][0], coin_meta["description"]["en"][:300]

('http://www.bitcoin.org',
 'Bitcoin is the first successful internet money based on peer-to-peer technology; whereby no central bank or authority is involved in the transaction and production of the Bitcoin currency. It was created by an anonymous individual/group under the name, Satoshi Nakamoto. The source code is available ')

# 8. Summary

This notebook demonstrated key aspects of working with the CoinGecko API:

- Accessing raw endpoints with `requests`
- Simplifying calls with reusable wrapper functions
- Exploring useful endpoints like:
  - `/coins/list` for supported assets
  - `/simple/price` for real-time quotes
  - `/coins/{id}` for metadata and descriptions

These tools form the data ingestion layer of our project, powering downstream forecasting and SHAP analysis in `SHAP.example.ipynb`.

For documentation of each wrapper, see `SHAP.API.md`.